# Phase 4 — Modelling
**UCI Hydraulic Systems Dataset — Predictive Maintenance**

Steps covered:
1. Load features and labels from Phase 3
2. Train/test split (stratified)
3. Baseline model — Logistic Regression
4. Mid-tier model — Random Forest
5. Primary model — XGBoost multi-output classifier
6. Hyperparameter tuning (RandomizedSearchCV)
7. Stratified 5-fold cross-validation
8. Evaluation — confusion matrices, classification reports, F1 comparison
9. Save best model


## 4.0 — Imports & configuration

In [13]:
from pathlib import Path
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.multioutput     import MultiOutputClassifier
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    RandomizedSearchCV, cross_val_score
)
from sklearn.preprocessing   import StandardScaler
from sklearn.pipeline        import Pipeline
from sklearn.metrics         import (
    f1_score, accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

# ── Paths ──────────────────────────────────────────────────────────
BASE_DIR      = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
MODELS_DIR    = BASE_DIR / 'models'
MODELS_DIR.mkdir(exist_ok=True)

# ── Label definitions (for display) ───────────────────────────────
TARGETS = ['cooler', 'valve', 'pump', 'accumulator']

LABEL_NAMES = {
    'cooler':      {0: 'near failure',    1: 'reduced eff.',   2: 'full eff.'},
    'valve':       {0: 'near failure',    1: 'severe lag',     2: 'small lag',      3: 'optimal'},
    'pump':        {0: 'no leakage',      1: 'weak leakage',   2: 'severe leakage'},
    'accumulator': {0: 'near failure',    1: 'severely red.',  2: 'slightly red.',  3: 'optimal'},
}

PALETTE = ['#7F77DD', '#1D9E75', '#EF9F27', '#D85A30']

print('Configuration loaded ✓')

Configuration loaded ✓


## 4.1 — Load features and labels

In [ ]:
X = pd.read_parquet(PROCESSED_DIR / 'features.parquet')
y = pd.read_csv(PROCESSED_DIR / 'labels_encoded.csv')

print(f'Features X: {X.shape}   (cycles × features)')
print(f'Labels   y: {y.shape}   (cycles × targets)')
print()

# Confirm alignment
assert X.shape[0] == y.shape[0], 'Row count mismatch between features and labels'
assert list(y.columns) == TARGETS, f'Unexpected label columns: {y.columns.tolist()}'

# Quick label distribution recap
print('Label distributions (encoded):')
for col in TARGETS:
    counts = y[col].value_counts().sort_index()
    dist   = '  '.join(f'cls{k}={v}' for k, v in counts.items())
    print(f'  {col:<14} {dist}')

ArrowKeyError: A type extension with name pandas.period already defined

## 4.2 — Train / test split

Stratified on all 4 targets simultaneously — we create a combined stratification key
so every combination of component health states is proportionally represented in both splits.
80/20 split, random_state fixed for reproducibility.

In [ ]:
RANDOM_STATE = 42
TEST_SIZE    = 0.20

# Stratification key: concatenate all 4 encoded labels into one string
# e.g. '0_1_2_3' — ensures proportional split across all class combinations
strat_key = y.astype(str).apply(lambda row: '_'.join(row), axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = TEST_SIZE,
    stratify     = strat_key,
    random_state = RANDOM_STATE,
)

# Reset indices
X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

print(f'Train set:  {X_train.shape[0]} cycles  ({100*(1-TEST_SIZE):.0f}%)')
print(f'Test set:   {X_test.shape[0]} cycles  ({100*TEST_SIZE:.0f}%)')
print()

# Verify stratification held — each class should appear in test set
print('Test set class coverage (all classes must appear):')
all_ok = True
for col in TARGETS:
    train_classes = set(y_train[col].unique())
    test_classes  = set(y_test[col].unique())
    missing = train_classes - test_classes
    if missing:
        print(f'  ⚠  {col}: missing classes {missing} in test set')
        all_ok = False
    else:
        counts = y_test[col].value_counts().sort_index()
        dist   = '  '.join(f'cls{k}={v}' for k, v in counts.items())
        print(f'  ✓  {col:<14} {dist}')

if all_ok:
    print('\nAll classes represented in test set ✓')

## 4.3 — Helper functions

In [ ]:
def evaluate_model(model, X_tr, y_tr, X_te, y_te, model_name):
    """
    Fit model, predict on test set, return a results dict with:
    - per-target accuracy and macro F1
    - overall mean macro F1
    - predictions (for confusion matrices)
    """
    model.fit(X_tr, y_tr)
    y_pred = pd.DataFrame(model.predict(X_te), columns=TARGETS)

    results = {'model_name': model_name, 'per_target': {}}

    f1_scores = []
    for col in TARGETS:
        acc = accuracy_score(y_te[col], y_pred[col])
        f1  = f1_score(y_te[col], y_pred[col], average='macro', zero_division=0)
        results['per_target'][col] = {'accuracy': acc, 'macro_f1': f1}
        f1_scores.append(f1)

    results['mean_macro_f1'] = np.mean(f1_scores)
    results['y_pred']        = y_pred
    return results


def print_results(results):
    name = results['model_name']
    print(f"{'─'*55}")
    print(f" {name}")
    print(f"{'─'*55}")
    for col in TARGETS:
        r   = results['per_target'][col]
        bar = '█' * int(r['macro_f1'] * 20)
        print(f"  {col:<14} acc={r['accuracy']:.3f}  macro_f1={r['macro_f1']:.3f}  {bar}")
    print(f"  {'MEAN':<14}                macro_f1={results['mean_macro_f1']:.3f}")
    print()


print('Helper functions defined ✓')

## 4.4 — Baseline: Logistic Regression

Simplest possible model. Sets the floor — anything below this means the other models have a bug.

In [ ]:
print('Training Logistic Regression baseline...\n')

# Logistic Regression needs feature scaling — wrap in a Pipeline
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    MultiOutputClassifier(
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    ))
])

lr_results = evaluate_model(lr_pipeline, X_train, y_train, X_test, y_test, 'Logistic Regression')
print_results(lr_results)

## 4.5 — Mid-tier: Random Forest

In [ ]:
print('Training Random Forest...\n')

# No scaling needed for tree-based models
rf_model = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators  = 200,
        max_depth     = None,     # Grow full trees
        min_samples_split = 2,
        random_state  = RANDOM_STATE,
        n_jobs        = -1,       # Use all CPU cores
    )
)

rf_results = evaluate_model(rf_model, X_train, y_train, X_test, y_test, 'Random Forest')
print_results(rf_results)

## 4.6 — Primary model: XGBoost (default params)

Run first with default params to get a baseline score before tuning.

In [ ]:
print('Training XGBoost (default params)...\n')

xgb_base = MultiOutputClassifier(
    XGBClassifier(
        n_estimators      = 200,
        max_depth         = 6,
        learning_rate     = 0.1,
        subsample         = 0.8,
        colsample_bytree  = 0.8,
        use_label_encoder = False,
        eval_metric       = 'mlogloss',
        random_state      = RANDOM_STATE,
        verbosity         = 0,
        n_jobs            = -1,
    )
)

xgb_base_results = evaluate_model(xgb_base, X_train, y_train, X_test, y_test, 'XGBoost (default)')
print_results(xgb_base_results)

## 4.7 — XGBoost hyperparameter tuning

RandomizedSearchCV with stratified 5-fold CV. Optimises on macro F1 for the cooler target
(as a representative proxy — we have 4 outputs, sklearn CV scores one at a time).
Runs 30 random combinations — fast enough on this dataset size.

In [ ]:
print('Tuning XGBoost with RandomizedSearchCV (30 iterations × 5 folds)...')
print('This will take 1-3 minutes...\n')

param_dist = {
    'n_estimators':     [100, 200, 300, 500],
    'max_depth':        [3, 4, 5, 6, 8],
    'learning_rate':    [0.01, 0.05, 0.1, 0.2, 0.3],
    'subsample':        [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma':            [0, 0.1, 0.2, 0.5],
    'reg_alpha':        [0, 0.01, 0.1, 1.0],
    'reg_lambda':       [0.5, 1.0, 2.0, 5.0],
}

# Run search on cooler target as representative proxy
search = RandomizedSearchCV(
    estimator  = XGBClassifier(
        use_label_encoder = False,
        eval_metric       = 'mlogloss',
        random_state      = RANDOM_STATE,
        verbosity         = 0,
        n_jobs            = -1,
    ),
    param_distributions = param_dist,
    n_iter              = 30,
    cv                  = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring             = 'f1_macro',
    random_state        = RANDOM_STATE,
    n_jobs              = -1,
    verbose             = 1,
)

search.fit(X_train, y_train['cooler'])

best_params = search.best_params_
print(f'\nBest params found:')
for k, v in best_params.items():
    print(f'  {k:<22} {v}')
print(f'\nBest CV macro F1 (cooler): {search.best_score_:.4f}')

## 4.8 — XGBoost with best params (tuned model)

In [ ]:
print('Training XGBoost with tuned hyperparameters...\n')

xgb_tuned = MultiOutputClassifier(
    XGBClassifier(
        **best_params,
        use_label_encoder = False,
        eval_metric       = 'mlogloss',
        random_state      = RANDOM_STATE,
        verbosity         = 0,
        n_jobs            = -1,
    )
)

xgb_tuned_results = evaluate_model(xgb_tuned, X_train, y_train, X_test, y_test, 'XGBoost (tuned)')
print_results(xgb_tuned_results)

## 4.9 — Stratified 5-fold cross-validation on tuned XGBoost

More reliable accuracy estimate than a single train/test split.
Report mean ± std of macro F1 across 5 folds per target.

In [ ]:
print('Running stratified 5-fold CV on full dataset with tuned XGBoost...\n')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = {}
for col in TARGETS:
    xgb_cv = XGBClassifier(
        **best_params,
        use_label_encoder = False,
        eval_metric       = 'mlogloss',
        random_state      = RANDOM_STATE,
        verbosity         = 0,
        n_jobs            = -1,
    )
    scores = cross_val_score(
        xgb_cv, X, y[col],
        cv      = cv,
        scoring = 'f1_macro',
        n_jobs  = -1,
    )
    cv_results[col] = scores
    bar = '█' * int(scores.mean() * 20)
    print(f'  {col:<14}  mean={scores.mean():.4f}  std=±{scores.std():.4f}  {bar}')
    print(f'               folds: {" ".join(f"{s:.3f}" for s in scores)}')
    print()

overall_mean = np.mean([cv_results[c].mean() for c in TARGETS])
print(f'  Overall mean macro F1 (5-fold CV): {overall_mean:.4f}')

## 4.10 — Model comparison table

In [ ]:
all_results = [lr_results, rf_results, xgb_base_results, xgb_tuned_results]

print(f'\n{"Model":<25}  {"Cooler":>8}  {"Valve":>8}  {"Pump":>8}  {"Accum.":>8}  {"Mean F1":>8}')
print('─' * 75)

for r in all_results:
    f1s  = [r['per_target'][c]['macro_f1'] for c in TARGETS]
    mean = r['mean_macro_f1']
    print(f"{r['model_name']:<25}  " +
          '  '.join(f'{f:.4f}' for f in f1s) +
          f'  {mean:.4f}')

print()
print(f'Note: XGBoost 5-fold CV (more reliable):')
cv_f1s = [cv_results[c].mean() for c in TARGETS]
print(f"  {'XGBoost (5-fold CV)':<25}  " +
      '  '.join(f'{f:.4f}' for f in cv_f1s) +
      f'  {np.mean(cv_f1s):.4f}')

## 4.11 — Diagnostic plots

In [ ]:
# ── Plot 1: Model comparison bar chart ────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

model_names = [r['model_name'] for r in all_results]
x           = np.arange(len(TARGETS))
width       = 0.18
offsets     = np.linspace(-0.27, 0.27, len(all_results))
bar_colors  = ['#888888', '#1D9E75', '#EF9F27', '#7F77DD']

for i, (r, color, offset) in enumerate(zip(all_results, bar_colors, offsets)):
    f1s  = [r['per_target'][c]['macro_f1'] for c in TARGETS]
    bars = ax.bar(x + offset, f1s, width, label=r['model_name'],
                  color=color, alpha=0.85, edgecolor='#1a1a2e', linewidth=0.5)
    for bar, f1 in zip(bars, f1s):
        if f1 > 0.05:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{f1:.2f}', ha='center', va='bottom', color='white',
                    fontsize=7, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels([t.capitalize() for t in TARGETS], color='#888', fontsize=10)
ax.set_ylabel('Macro F1 Score', color='#888', fontsize=10)
ax.set_ylim(0, 1.12)
ax.set_title('Model comparison — macro F1 per target', color='white', fontsize=12, pad=10)
ax.tick_params(colors='#888')
for spine in ax.spines.values():
    spine.set_edgecolor('#30363d')
ax.axhline(y=1.0, color='#30363d', linestyle='--', linewidth=0.8, alpha=0.5)
ax.legend(fontsize=9, labelcolor='white', facecolor='#161b22',
          edgecolor='#30363d', loc='upper left')

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'plot_model_comparison.png',
            dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ── Plot 2: Cross-validation scores per target ────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
fig.patch.set_facecolor('#0d1117')

for ax, col, color in zip(axes, TARGETS, PALETTE):
    scores = cv_results[col]
    folds  = np.arange(1, 6)

    ax.set_facecolor('#161b22')
    ax.bar(folds, scores, color=color, alpha=0.8,
           edgecolor='#1a1a2e', linewidth=0.5)
    ax.axhline(scores.mean(), color='white', linewidth=1.2,
               linestyle='--', label=f'Mean={scores.mean():.3f}')
    ax.fill_between(
        [0.5, 5.5],
        scores.mean() - scores.std(),
        scores.mean() + scores.std(),
        alpha=0.15, color='white'
    )

    for fold, score in zip(folds, scores):
        ax.text(fold, score + 0.002, f'{score:.3f}',
                ha='center', color='white', fontsize=8)

    ax.set_title(col.capitalize(), color='white', fontsize=11)
    ax.set_xlabel('Fold', color='#888', fontsize=9)
    ax.set_ylabel('Macro F1' if col == 'cooler' else '', color='#888', fontsize=9)
    ax.set_ylim(0.7, 1.05)
    ax.set_xticks(folds)
    ax.tick_params(colors='#888', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363d')
    ax.legend(fontsize=8, labelcolor='white', facecolor='#161b22',
              edgecolor='#30363d')

fig.suptitle('XGBoost tuned — stratified 5-fold CV macro F1 per target',
             color='white', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'plot_cv_scores.png',
            dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ── Plot 3: Confusion matrices for tuned XGBoost ─────────────────
y_pred_tuned = xgb_tuned_results['y_pred']

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.patch.set_facecolor('#0d1117')

for ax, col, color in zip(axes, TARGETS, PALETTE):
    cm     = confusion_matrix(y_test[col], y_pred_tuned[col])
    labels = [LABEL_NAMES[col][i] for i in sorted(LABEL_NAMES[col].keys())
              if i in y_test[col].unique()]
    n_cls  = cm.shape[0]

    ax.set_facecolor('#161b22')

    # Normalise by row (true class) for percentage display
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1, aspect='auto')

    for i in range(n_cls):
        for j in range(n_cls):
            pct = cm_norm[i, j]
            ax.text(j, i, f'{cm[i,j]}\n({pct:.0%})',
                    ha='center', va='center', fontsize=8,
                    color='white' if pct > 0.5 else '#aaa')

    tick_labels = [l.replace(' ', '\n') for l in labels]
    ax.set_xticks(range(n_cls))
    ax.set_yticks(range(n_cls))
    ax.set_xticklabels(tick_labels, color='#888', fontsize=7)
    ax.set_yticklabels(tick_labels, color='#888', fontsize=7)
    ax.set_xlabel('Predicted', color='#888', fontsize=9)
    ax.set_ylabel('Actual', color='#888', fontsize=9)
    ax.set_title(col.capitalize(), color='white', fontsize=11, pad=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363d')

fig.suptitle('XGBoost (tuned) — confusion matrices on test set (row-normalised)',
             color='white', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'plot_confusion_matrices.png',
            dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ── Full classification report for tuned XGBoost ──────────────────
print('XGBoost (tuned) — full classification report on test set\n')
for col in TARGETS:
    label_list = [LABEL_NAMES[col][i] for i in sorted(LABEL_NAMES[col].keys())
                  if i in y_test[col].unique()]
    print(f'  {col.upper()}')
    print(classification_report(
        y_test[col], y_pred_tuned[col],
        target_names = label_list,
        zero_division = 0
    ))

## 4.12 — Save best model and results

In [ ]:
# Save tuned XGBoost model
model_path = MODELS_DIR / 'xgb_tuned.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(xgb_tuned, f)

# Save best params
with open(MODELS_DIR / 'best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)

# Save results summary for dashboard
results_summary = {
    'train_size':     int(X_train.shape[0]),
    'test_size':      int(X_test.shape[0]),
    'n_features':     int(X.shape[1]),
    'random_state':   RANDOM_STATE,
    'models': {
        r['model_name']: {
            'mean_macro_f1': round(r['mean_macro_f1'], 4),
            'per_target': {
                col: {
                    'accuracy': round(r['per_target'][col]['accuracy'], 4),
                    'macro_f1': round(r['per_target'][col]['macro_f1'], 4),
                }
                for col in TARGETS
            }
        }
        for r in all_results
    },
    'cv_results': {
        col: {
            'mean':   round(float(cv_results[col].mean()), 4),
            'std':    round(float(cv_results[col].std()),  4),
            'folds':  [round(float(s), 4) for s in cv_results[col]]
        }
        for col in TARGETS
    }
}

with open(MODELS_DIR / 'results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

# Save test set predictions for dashboard
y_pred_tuned.to_csv(MODELS_DIR / 'y_pred_test.csv', index=False)
y_test.to_csv(MODELS_DIR / 'y_test.csv', index=False)
X_test.to_parquet(MODELS_DIR / 'X_test.fastparquet', index=False)

print('Saved to models/:\n')
print('  xgb_tuned.pkl         — trained model')
print('  best_params.json      — tuned hyperparameters')
print('  results_summary.json  — all model scores')
print('  y_pred_test.csv       — test set predictions')
print('  y_test.csv            — test set ground truth')
print('  X_test.parquet        — test set features (for SHAP)')

## 4.13 — Summary

| Model | Cooler | Valve | Pump | Accumulator | Mean F1 |
|---|---|---|---|---|---|
| Logistic Regression | — | — | — | — | (see output) |
| Random Forest | — | — | — | — | (see output) |
| XGBoost default | — | — | — | — | (see output) |
| **XGBoost tuned** | — | — | — | — | **(see output)** |
| XGBoost 5-fold CV | — | — | — | — | (see output) |

**Next → `05_shap_explainability.ipynb`**

Compute SHAP values to identify which sensors and features drive each prediction.
Generate beeswarm plot, feature importance bar chart, and per-cycle waterfall explanations.